# 🤖 Agentic Statistical Assistant
### From RAG to Agent — **interactive edition** · STG17 Workshop · AfDB / STATAFRIC · Day 1

> **In one sentence:** you will build, piece by piece, an assistant that answers users’ questions from a statistical office’s publications — then use it through a **chat interface**, with a trace of every decision, a human approval queue and an audit log.

| ⏱ Duration | 🆓 Cost | 🔑 API key | 🇨🇮 Case study | 🧩 Dependencies |
|:--:|:--:|:--:|:--:|:--:|
| 90 min | free | optional | Côte d’Ivoire · **fictional** data | no in-house library |

---

## 🧭 How to use this notebook

1. **In Colab:** menu **Runtime → Run all** (or `Ctrl + F9`). Allow about 30 seconds.
2. **Scroll down to Step 10**: the chat interface appears. Ask your questions.
3. **Then scroll back up** to understand each building block: every step explains *why* it exists.

> 💡 **No key is required.** By default, the interface uses a local “demo brain” that speaks exactly the same language as a real model. For a real **free** model, add a Groq key (Step 8).

---

## 🗺️ Roadmap

| Part | Step | Content | What you learn |
|:--|:--:|:--|:--|
| **🟦 Prepare** | 1 | Setup and display helpers | A reproducible environment |
| | 2 | The corpus | Working with controlled sources |
| **🟩 Retrieve** | 3 | Two search engines, measured | Why approach B beats approach A |
| **🟨 Act** | 4 | The tools | Read, compute, write: three levels of risk |
| | 5 | The protocol | How a model *requests* an action |
| | 6 | Policies and approval queue | Where the human stays in control |
| | 7 | The loop, grounding and audit | The core of the agent and its controls |
| **🟪 Think** | 8 | The “brains” | Local demo, Groq, Gemini or Ollama |
| | 9 | Text-mode trial | Check before opening the interface |
| **🟥 Use** | 10 | **The chat interface** | An assistant your colleagues can use |
| | 11 | Under the hood | What happens when you click “Send” |
| **⬛ Wrap up** | 12 | Limits, exercises, troubleshooting | From lab to production |

---
# 🟦 PART 1 — Prepare

## Step 1 · Setup and display helpers

**🎯 Goal:** give every participant the same environment.

This cell installs the standard libraries if needed (`scikit-learn`, `pandas`, `matplotlib`, `ipywidgets`) and defines the **display style**. Every visual element is produced by Python code: this is what guarantees identical rendering in **Colab**, **Jupyter** and **VS Code**.

In [ ]:
# ── Step 1 · Setup and configuration ────────────────────────────────────────
import importlib, subprocess, sys

for module, package in [("sklearn", "scikit-learn>=1.3"), ("pandas", "pandas>=2.0"), ("matplotlib", "matplotlib>=3.7"),
                        ("ipywidgets", "ipywidgets>=7.6"), ("requests", "requests")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)

import ast, hashlib, html, json, operator, os, re, shutil, time, unicodedata
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Lab context
COUNTRY = {"iso3": "CIV", "name": "Côte d’Ivoire"}
ROOT = Path("stg17_lab")
CORPUS_DIR = ROOT / "corpus_en"
OUTPUTS = ROOT / "outputs" / COUNTRY["iso3"]
for folder in (CORPUS_DIR, OUTPUTS):
    folder.mkdir(parents=True, exist_ok=True)

# Colour palette
GREEN, NAVY, GOLD, RED, GREY, LIGHT = "#1B7A43", "#0B2545", "#F2A900", "#B83B2E", "#6B7B75", "#EAF5EE"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10.5, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9AA8A1", "axes.titleweight": "bold", "axes.titlesize": 12.5,
})

# ── Display helpers ─────────────────────────────────────────────────────────
_STYLES = {
    "info":    (GREEN, "#EAF5EE", "💡"),
    "key":     (NAVY,  "#E8EEF6", "🔑"),
    "warning": (GOLD,  "#FFF7E0", "⚠️"),
    "danger":  (RED,   "#FBECEA", "⛔"),
    "success": (GREEN, "#E3F4E9", "✅"),
}

def callout(text, title="", kind="info"):
    """Show a coloured callout box (info, key, warning, danger, success)."""
    colour, background, icon = _STYLES[kind]
    display(HTML(
        f"<div style='border-left:5px solid {colour};background:{background};padding:11px 16px;border-radius:8px;"
        f"margin:8px 0;font-family:Segoe UI,system-ui,sans-serif;color:#1d2b24;line-height:1.5'>"
        f"<b>{icon} {title}</b><div style='margin-top:3px'>{text}</div></div>"))

_TABLE_CSS = ("<style>.stg{border-collapse:collapse;font-family:Segoe UI,system-ui,sans-serif;font-size:12.5px;margin:6px 0}"
              ".stg th{background:#0B2545;color:#fff;padding:7px 10px;text-align:left}"
              ".stg td{padding:6px 10px;border-bottom:1px solid #DDE3E0;vertical-align:top}"
              ".stg tr:nth-child(even) td{background:#F4F7F5}</style>")

def show_table(df, caption=None, max_width=90):
    """Display a DataFrame in the lab style."""
    d = df.copy()
    for col in d.columns:
        if d[col].dtype == object:
            d[col] = d[col].map(lambda v: (str(v)[:max_width] + "…") if isinstance(v, str) and len(v) > max_width else v)
    block = _TABLE_CSS + d.to_html(index=False, classes="stg", border=0, escape=True)
    if caption:
        block += f"<div style='color:{GREY};font-size:12px;font-style:italic;margin:2px 0 10px'>{caption}</div>"
    display(HTML(block))

def banner(overline, title, subtitle, colour=NAVY):
    """Section banner, rendered as HTML from Python (reliable in Colab)."""
    display(HTML(
        f"<div style='background:linear-gradient(135deg,{colour} 0%,#1B7A43 100%);border-radius:16px;padding:22px 30px;"
        f"font-family:Segoe UI,system-ui,sans-serif;margin:6px 0'>"
        f"<div style='color:#F2A900;font-size:12px;letter-spacing:3px;font-weight:700'>{overline}</div>"
        f"<div style='color:#fff;font-size:26px;font-weight:800;margin:6px 0'>{title}</div>"
        f"<div style='color:#dbe7e0;font-size:14.5px;line-height:1.5'>{subtitle}</div></div>"))

banner("STG17 WORKSHOP · AfDB / STATAFRIC · DAY 1", "🤖 Agentic Statistical Assistant",
       "Find the right information · act under control · trace everything — "
       "<b style='color:#fff'>the model asks, your code executes.</b>")

callout(f"Working folder: <code>{ROOT}/</code><br>Outputs: <code>{OUTPUTS}</code>", "Environment ready", "success")

## Step 2 · The study corpus

**🎯 Goal:** work with controlled sources whose answers are all known.

Nine **fictional** publications from a national statistical office: employment, prices, national accounts, census, education, cocoa, health, poverty, and a methodological note.

> ⚠️ **All figures are invented.** They look like real statistics but must never be quoted.

**🔍 What to notice:** the corpus is deliberately tricky. Several documents mention “rates” and “unemployment”, and some sentences do not name their subject (“It stands at…”).

In [ ]:
# ── Step 2 · The corpus: nine fictional publications ────────────────────────
CORPUS = {
"employment_2023Q4": ("National Labour Force Survey — Fourth quarter 2023", """
In the fourth quarter of 2023, the ILO unemployment rate stood at 8.6% of the labour force.
Youth unemployment is much higher among people aged 15 to 24. For this age group, it reaches 19.2%, more than double the national average.
Women remain more exposed than men. Their unemployment rate is 10.1%, compared with 7.4% for men.
The participation rate of the working-age population stands at 61.3%.
Informal employment remains predominant. It accounts for 88.5% of total employment, with a higher share in rural areas.
"""),
"cpi_2024_03": ("Harmonised Consumer Price Index — March 2024", """
In March 2024, year-on-year inflation came in at 4.1%, after 4.5% in February.
The price increase was driven mainly by food and non-alcoholic beverages, whose prices rose by 6.3% over one year.
Transport prices rose by 2.2% and those of housing, water and electricity by 3.0%.
Core inflation, which excludes fresh products and energy, stood at 3.4%.
"""),
"national_accounts_2023": ("Annual National Accounts — 2023", """
The economy recorded real gross domestic product growth of 6.5% in 2023.
The primary sector contributes 17% of value added, industry 23% and services 50%.
It was driven by financial services and telecommunications, which grew by 9.8%.
Public investment accounted for 7.2% of GDP.
"""),
"census_2021": ("General Population and Housing Census — 2021", """
The census counts 29.4 million inhabitants living in the national territory.
The population has grown at an average annual rate of 2.9% since the previous census.
More than half of the inhabitants now live in cities. The urbanisation rate reaches 52.5%.
The population is very young. Its median age is 18.9 years.
"""),
"education_2022": ("Statistical Yearbook of Education — 2022", """
The gross enrolment ratio in primary education is 101%, reflecting the presence of pupils outside the official age.
The primary completion rate stands at 81%.
Among adults aged 15 and over, the literacy rate is estimated at 53%.
There are on average 41 pupils per teacher in public primary schools.
"""),
"cocoa_2023": ("Economic Brief — cocoa sector, 2022-2023 season", """
National production of cocoa beans is estimated at 2.2 million tonnes for the 2022-2023 season.
The sector accounts for about 40% of the country’s export earnings.
The guaranteed farm-gate price paid to producers was set at 1,000 FCFA per kilogram.
"""),
"health_2022": ("Health Dashboard — 2022", """
Life expectancy at birth is estimated at 59 years.
The under-five mortality rate stands at 75 deaths per 1,000 live births.
DTP3 vaccination coverage among children aged 12 to 23 months reaches 84%.
"""),
"poverty_2021": ("Poverty Profile — 2021", """
The share of the population living below the national poverty line is 37.5%.
In rural areas it reaches 45%, compared with 25% in urban areas.
Consumption inequality, measured by the Gini index, stands at 0.35.
"""),
"methodology_employment": ("Methodological Note — Labour Force Survey definitions", """
A young person is defined as a person aged 15 to 24. Youth unemployment is measured according to the criteria of the International Labour Organization.
A person is considered unemployed if they are without work, available and actively seeking employment.
Informal employment covers undeclared jobs and unregistered production units.
The rates presented in quarterly publications are weighted and adjusted for non-response.
"""),
}

for doc_id, (title, text) in CORPUS.items():
    (CORPUS_DIR / f"{doc_id}.md").write_text(f"# {title}\n{text.strip()}\n", encoding="utf-8")

def load_corpus(folder):
    """Read each file: first line = title, the rest = text."""
    documents = []
    for path in sorted(Path(folder).glob("*.md")):
        lines = path.read_text(encoding="utf-8").splitlines()
        documents.append({"id": path.stem, "title": lines[0].lstrip("# ").strip(), "text": "\n".join(lines[1:]).strip()})
    return documents

DOCS = load_corpus(CORPUS_DIR)
show_table(pd.DataFrame([{"identifier": d["id"], "title": d["title"], "words": len(d["text"].split())} for d in DOCS]),
           f"{len(DOCS)} documents loaded from {CORPUS_DIR} — fictional data.")

---
# 🟩 PART 2 — Retrieve the information

## Step 3 · Two search engines, measured

**🎯 Goal:** understand that an assistant is only as good as the passages it retrieves.

A **RAG** (*Retrieval-Augmented Generation*) system first retrieves the relevant passages, then answers **only** from them. We compare two engines:

| | 🅰️ Approach A — naive | 🅱️ Approach B — improved |
|:--|:--|:--|
| **Chunking** | every 220 characters, even mid-word | whole sentences, with overlap |
| **Context** | none | the document title is added to every passage |
| **Matching** | exact words (“unemployement” ≠ “unemployment”) | lower-cased, accent-free text + 3-to-5-letter fragments |
| **Ranking** | TF-IDF only | BM25 **and** fragments, merged by rank (RRF) |

**📏 How we measure:** 13 questions with known answers. A passage counts as **relevant** if it contains both the expected figure and a clue word.

- **Hit@1** — share of questions whose first passage is relevant;
- **Hit@3** — share of questions with a relevant passage in the top 3;
- **MRR** — mean of 1 / rank of the first relevant passage (1 = perfect).

> 💡 In the interface (Step 10), you can **switch between engines** and see the difference on your own questions.

In [ ]:
# ── Step 3 · The two search engines ─────────────────────────────────────────
# 🅰️ Approach A: fixed-size chunks + word-level TF-IDF
def chunk_fixed(documents, size=220):
    """Cut each text every `size` characters, without overlap."""
    chunks = []
    for doc in documents:
        text = doc["text"].replace("\n", " ")
        for start in range(0, len(text), size):
            piece = text[start:start + size]
            chunks.append({"doc": doc["id"], "title": doc["title"], "text": piece, "context": piece})
    return chunks


class NaiveSearch:
    """Approach A — word-level TF-IDF on raw text."""
    name = "A · Naive"

    def __init__(self, chunks):
        self.chunks = chunks
        self.vectorizer = TfidfVectorizer()
        self.matrix = self.vectorizer.fit_transform([c["text"] for c in chunks])

    def search(self, question, k=3):
        scores = cosine_similarity(self.vectorizer.transform([question]), self.matrix).ravel()
        order = np.argsort(-scores)[:k]
        return [{"chunk": self.chunks[i], "score": float(scores[i]), "confidence": float(scores[i])} for i in order]


CHUNKS_A = chunk_fixed(DOCS)
ENGINE_A = NaiveSearch(CHUNKS_A)


# 📏 Evaluation set and metrics
# (question, expected figure, clue that must appear in the passage)
EVALUATION = [
    ("What is the youth unemployement rate?",            "19.2", "15 to 24"),
    ("What was annual inflation in March 2024?",         "4.1",  "March 2024"),
    ("How much did food prices increase?",               "6.3",  "food"),
    ("What was economic growth in 2023?",                "6.5",  "growth"),
    ("How many inhabitants does the country have?",      "29.4", "inhabitants"),
    ("What share of the population lives in cities?",    "52.5", "urbanisation"),
    ("What share of jobs is informal?",                  "88.5", "informal"),
    ("What is the primary school completion rate?",      "81",   "completion"),
    ("How much cocoa was produced?",                     "2.2",  "cocoa"),
    ("What is life expectancy at birth?",                "59",   "life expectancy"),
    ("What is the poverty level in rural areas?",        "45",   "rural"),
    ("What is children's vaccination coverage?",         "84",   "vaccination"),
    ("Is unemployment higher for women than for men?",   "10.1", "women"),
]

def _norm(text):
    """Lower-case and strip accents, to compare regardless of spelling."""
    text = unicodedata.normalize("NFKD", text.lower())
    return "".join(c for c in text if not unicodedata.combining(c))

def is_relevant(chunk, expected, clue):
    context = _norm(chunk["context"])
    return _norm(expected) in context and _norm(clue) in context

def format_rank(r):
    return "❌ not in top 3" if pd.isna(r) else f"✅ {int(r)}"

def evaluate(engine, k=3):
    rows = []
    for question, expected, clue in EVALUATION:
        results = engine.search(question, k=k)
        rank = next((r + 1 for r, res in enumerate(results) if is_relevant(res["chunk"], expected, clue)), None)
        rows.append({"question": question, "expected": expected, "rank": rank})
    detail = pd.DataFrame(rows)
    summary = {
        "engine": engine.name,
        "Hit@1": (detail["rank"] == 1).mean(),
        "Hit@3": detail["rank"].notna().mean(),
        "MRR": detail["rank"].map(lambda r: 0.0 if pd.isna(r) else 1 / r).mean(),
    }
    return summary, detail


# 🅱️ Approach B: improved retrieval pipeline
STOP_WORDS = set("""the a an of to in on for and or is are was were be been being what which who whom how much many
does do did has have had it its this that these those with as at by from than their there more most per over
into about i we you they he she can could would should""".split())

def tokenize(text):
    """Normalise, then split into useful words (no stop words, simple plural removed)."""
    words = re.findall(r"[a-z0-9]+", _norm(text))
    return [w[:-1] if len(w) > 4 and w.endswith("s") else w for w in words if w not in STOP_WORDS]


# ① + ② Sentence-based chunking, with overlap and a contextual header
def chunk_sentences(documents, max_chars=260, overlap=1):
    """Group whole sentences; the last sentence of a chunk opens the next one."""
    chunks = []
    for doc in documents:
        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", doc["text"].replace("\n", " ")) if s.strip()]
        groups, current = [], []
        for sentence in sentences:
            if current and len(" ".join(current + [sentence])) > max_chars:
                groups.append(current)
                current = current[-overlap:] if overlap else []
            current.append(sentence)
        if current:
            groups.append(current)
        for group in groups:
            text = " ".join(group)
            chunks.append({"doc": doc["id"], "title": doc["title"], "text": text,
                           "context": f"[{doc['title']}] {text}"})        # ② contextual header
    return chunks


# ④ BM25, implemented in a few lines
class BM25:
    def __init__(self, tokenized_docs, k1=1.5, b=0.75):
        self.docs, self.k1, self.b = tokenized_docs, k1, b
        self.lengths = np.array([len(d) for d in tokenized_docs], dtype=float)
        self.avg_length = self.lengths.mean()
        n = len(tokenized_docs)
        doc_freq = {}
        for d in tokenized_docs:
            for word in set(d):
                doc_freq[word] = doc_freq.get(word, 0) + 1
        self.idf = {w: np.log(1 + (n - f + 0.5) / (f + 0.5)) for w, f in doc_freq.items()}

    def scores(self, query):
        result = np.zeros(len(self.docs))
        for i, doc in enumerate(self.docs):
            counts = {}
            for word in doc:
                counts[word] = counts.get(word, 0) + 1
            for word in query:
                if word in counts:
                    tf = counts[word]
                    norm = self.k1 * (1 - self.b + self.b * self.lengths[i] / self.avg_length)
                    result[i] += self.idf[word] * tf * (self.k1 + 1) / (tf + norm)
        return result


class HybridSearch:
    """Approach B — BM25 (words) + character n-gram TF-IDF, merged with RRF."""
    name = "B · Improved"

    def __init__(self, chunks, k_rrf=60):
        self.chunks, self.k_rrf = chunks, k_rrf
        texts = [c["context"] for c in chunks]                          # ② the header is indexed
        self.bm25 = BM25([tokenize(t) for t in texts])                  # ④
        self.vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)  # ③
        self.matrix = self.vectorizer.fit_transform([_norm(t) for t in texts])

    def search(self, question, k=3):
        s_bm25 = self.bm25.scores(tokenize(question))
        s_ngram = cosine_similarity(self.vectorizer.transform([_norm(question)]), self.matrix).ravel()
        rrf = np.zeros(len(self.chunks))
        for scores in (s_bm25, s_ngram):                                # ④ rank fusion
            for rank, i in enumerate(np.argsort(-scores)):
                rrf[i] += 1 / (self.k_rrf + rank + 1)
        order = np.argsort(-rrf)[:k]
        return [{"chunk": self.chunks[i], "score": float(rrf[i]), "confidence": float(s_ngram[i])} for i in order]


CHUNKS_B = chunk_sentences(DOCS)
ENGINE_B = HybridSearch(CHUNKS_B)


# 🛑 Knowing when to say “I don’t know”: a two-part rule
COSINE_THRESHOLD = 0.30     # n-gram similarity considered sufficient on its own
COVERAGE_THRESHOLD = 0.60   # or: share of the question's words found in the passage
INSTRUCTION_WORDS = {"prepare", "draft", "write", "note", "memo", "give", "show", "tell", "level", "please", "summary",
                     "exceed", "overall", "compare", "difference", "gap", "higher", "lower", "between"}   # instruction & comparison words

def coverage(question, chunk):
    """Share of the question's useful words present in the passage (compared on 5 letters)."""
    words = [w for w in tokenize(question) if w not in INSTRUCTION_WORDS and len(w) > 2]
    if not words:
        return 0.0
    prefixes = {w[:5] for w in tokenize(chunk["context"])}
    return sum(w[:5] in prefixes for w in words) / len(words)

def passage_is_sufficient(question, result):
    """True if the best passage is close enough OR covers most of the question."""
    cov = coverage(question, result["chunk"])
    return (result["confidence"] >= COSINE_THRESHOLD or cov >= COVERAGE_THRESHOLD), cov


# 🧾 Grounded answer (used by the “Simple RAG” mode of the interface)
RAG_INSTRUCTIONS = """You are the documentation assistant of a national statistical office.
Mandatory rules:
1. Answer ONLY from the excerpts provided.
2. Cite the source of every figure in square brackets, e.g. [employment_2023Q4].
3. If the excerpts do not contain the answer, reply exactly: "Information not found in the corpus."
4. Do not round or recompute figures."""

def build_prompt(question, results):
    excerpts = "\n".join(f"[{r['chunk']['doc']}] {r['chunk']['context']}" for r in results)
    return f"EXCERPTS:\n{excerpts}\n\nQUESTION: {question}"

def extractive_answer(question, results):
    """Without a model: return the best passage, word for word, with its source."""
    best = results[0]["chunk"]
    return f"“{best['text']}” [{best['doc']}]"

def answer(question, engine=None, generator=None):
    engine = engine or ENGINE_B
    results = engine.search(question, k=3)
    if not passage_is_sufficient(question, results[0])[0]:
        return {"question": question, "confidence": results[0]["confidence"], "mode": "refusal (rule)",
                "answer": "Information not found in the corpus."}
    if generator is None:
        return {"question": question, "confidence": results[0]["confidence"], "mode": "extractive",
                "answer": extractive_answer(question, results)}
    text = generator(RAG_INSTRUCTIONS, [{"role": "user", "content": build_prompt(question, results)}])
    return {"question": question, "confidence": results[0]["confidence"], "mode": "LLM", "answer": text}


ENGINES = {"B · Improved": ENGINE_B, "A · Naive": ENGINE_A}
callout(f"Approach A: <b>{len(CHUNKS_A)}</b> passages · Approach B: <b>{len(CHUNKS_B)}</b> passages · "
        f"refusal if cosine < <b>{COSINE_THRESHOLD}</b> and coverage < <b>{COVERAGE_THRESHOLD}</b>", "Engines ready", "success")

In [ ]:
# ── Step 3 (continued) · The comparison, in one chart ───────────────────────
SUMMARY_A, DETAIL_A = evaluate(ENGINE_A)
SUMMARY_B, DETAIL_B = evaluate(ENGINE_B)

fig, ax = plt.subplots(figsize=(8.8, 3.4))
x = np.arange(3)
for j, (s, colour) in enumerate([(SUMMARY_A, "#A9B5B0"), (SUMMARY_B, GREEN)]):
    values = [s["Hit@1"], s["Hit@3"], s["MRR"]]
    bars = ax.bar(x + (j - 0.5) * 0.36, values, 0.34, color=colour, label=s["engine"])
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=10, fontweight="bold")
ax.set_xticks(x, ["Hit@1", "Hit@3", "MRR"]); ax.set_ylim(0, 1.28); ax.set_yticks([0, .25, .5, .75, 1])
ax.set_title("Retrieval quality on 13 questions (higher = better)")
ax.legend(frameon=False, loc="upper center", ncol=2)
plt.tight_layout(); plt.show()

comparison = pd.DataFrame({"question": DETAIL_A["question"],
                           "🅰️ Naive": DETAIL_A["rank"].map(format_rank),
                           "🅱️ Improved": DETAIL_B["rank"].map(format_rank)})
show_table(comparison, "Rank of the first relevant passage, question by question.")
callout(f"MRR: <b>{SUMMARY_A['MRR']:.2f} → {SUMMARY_B['MRR']:.2f}</b> · Hit@1: <b>{SUMMARY_A['Hit@1']:.0%} → {SUMMARY_B['Hit@1']:.0%}</b>. "
        "No additional AI model: just better chunking, better normalisation and better combination. "
        "With so few questions, these differences are indicative only.", "Key takeaway", "key")

### 🛑 Knowing when to say “I don’t know”

A trustworthy assistant must **refuse** when the corpus does not contain the answer. A single similarity threshold is not enough: short questions (“inflation”) get a low score even though they are legitimate, while some off-topic questions (“Who is the minister of the economy?”) score high thanks to one shared word.

So we combine **two criteria**. A passage is considered sufficient if:

- its **similarity** to the question is high (cosine ≥ 0.30), **or**
- it **covers** at least 60% of the question’s useful words.

**🔍 What to notice:** the table shows the decision for legitimate and out-of-corpus questions. No rule is perfect: look for any case that slips through, and remember there are two more lines of defence (the model’s instructions and human review).

In [ ]:
# ── Step 3 (continued) · Refusal, question by question ─────────────────────
refusal_trials = [("What was inflation in March 2024?", "legitimate"), ("inflation", "legitimate"),
                  ("How many inhabitants?", "legitimate"), ("life expectency", "legitimate"),
                  ("What is the US dollar exchange rate?", "off-corpus"), ("What is the price of oil?", "off-corpus"),
                  ("Who is the minister of the economy?", "off-corpus"), ("What is the maternal mortality rate?", "off-corpus")]
rows, mistakes = [], []
for q, nature in refusal_trials:
    best = ENGINE_B.search(q)[0]
    ok, cov = passage_is_sufficient(q, best)
    correct = ok == (nature == "legitimate")
    if not correct:
        mistakes.append(q)
    rows.append({"question": q, "nature": nature, "cosine": f"{best['confidence']:.2f}", "coverage": f"{cov:.0%}",
                 "decision": "✅ answers" if ok else "🛑 refuses", "verdict": "👍" if correct else "⚠️ mistake"})
show_table(pd.DataFrame(rows),
           (f"Mistakes: {', '.join(mistakes)} — hence rule 3 of the instructions and human review."
            if mistakes else "All decisions are correct on this sample — which does not prove the rule is perfect."))

---
# 🟨 PART 3 — Act, under control

## Step 4 · The tools

**🎯 Goal:** define precisely what the agent is allowed to *request*.

| Tool | Type | Role | Risk |
|:--|:--:|:--|:--|
| `search_documents` | 👁️ read | queries engine B | low |
| `list_documents` | 👁️ read | inventory of publications | low |
| `calculate` | 👁️ read | **safe** arithmetic | low — if the code is protected |
| `save_note` | ✍️ **write** | creates a file | **real**: subject to approval |

**🧠 Three design choices to remember:**

- `writes` is a **field** of the tool, not a comment: the policy cannot forget it.
- The calculator **parses** the expression instead of executing it: a model cannot make it run code.
- The search tool **refuses by itself** when no passage is relevant enough: the tool says “I don’t know” before the model improvises.

In [ ]:
# ── Step 4 · The tools ──────────────────────────────────────────────────────
@dataclass
class Tool:
    name: str
    description: str
    parameters: dict               # parameter name -> description
    function: Callable
    writes: bool = False           # ← the property that matters

    def run(self, **arguments):
        unknown = set(arguments) - set(self.parameters)
        if unknown:
            raise TypeError(f"unknown parameter(s): {sorted(unknown)}; expected: {sorted(self.parameters)}")
        return str(self.function(**arguments))


def search_tool(engine):
    def search_documents(query):
        results = engine.search(str(query), k=3)
        if not passage_is_sufficient(str(query), results[0])[0]:
            return "NO RELEVANT PASSAGE: the information seems to be missing from the corpus."
        return "\n".join(f"[{r['chunk']['doc']}] {r['chunk']['text']}" for r in results)
    return Tool("search_documents", "Searches passages in the office's publications.",
                {"query": "keywords or question"}, search_documents)


def list_tool(documents):
    def list_documents():
        lines = [f"{d['id']} — {d['title']}" for d in documents]
        return f"{len(lines)} publications available:\n" + "\n".join(lines)
    return Tool("list_documents", "Lists the available publications.", {}, list_documents)


# Safe calculator: the expression is parsed, never executed
_OPERATIONS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
               ast.Div: operator.truediv, ast.Pow: operator.pow}
_UNARY = {ast.USub: operator.neg, ast.UAdd: operator.pos}

def safe_calculation(expression):
    expression = str(expression).replace(",", "").strip()      # "1,000" -> "1000"
    if len(expression) > 80:
        raise ValueError("expression too long")
    def evaluate_node(n):
        if isinstance(n, ast.Expression):
            return evaluate_node(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)) and not isinstance(n.value, bool):
            return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPERATIONS:
            left, right = evaluate_node(n.left), evaluate_node(n.right)
            if isinstance(n.op, ast.Pow) and abs(right) > 10:
                raise ValueError("exponent too large")
            return _OPERATIONS[type(n.op)](left, right)
        if isinstance(n, ast.UnaryOp) and type(n.op) in _UNARY:
            return _UNARY[type(n.op)](evaluate_node(n.operand))
        raise ValueError(f"forbidden element: {type(n).__name__}")
    return round(evaluate_node(ast.parse(expression, mode="eval")), 6)

def calculator_tool():
    def calculate(expression):
        try:
            return safe_calculation(expression)
        except (ValueError, SyntaxError, ZeroDivisionError) as error:
            return f"ERROR: {error}"
    return Tool("calculate", "Performs an arithmetic calculation (+ - * / **).", {"expression": "e.g. 19.2 - 8.6"}, calculate)


# Note writer: the file name is sanitised (no escape from the authorised folder)
def note_tool(folder):
    folder = Path(folder)
    def save_note(file_name, text):
        name = Path(str(file_name)).name
        if not re.fullmatch(r"[\w\-]{1,60}\.md", name):
            return f"ERROR: file name rejected ({file_name!r})"
        folder.mkdir(parents=True, exist_ok=True)
        (folder / name).write_text(str(text), encoding="utf-8")
        return f"note saved: {name}"
    return Tool("save_note", "Saves a summary note (.md file).",
                {"file_name": "e.g. finding.md", "text": "content of the note"}, save_note, writes=True)


NOTES_DIR = OUTPUTS / "notes"
shutil.rmtree(NOTES_DIR, ignore_errors=True)          # start from an empty folder on every run

def build_tools(engine):
    """The four tools, connected to the chosen search engine."""
    return [search_tool(engine), list_tool(DOCS), calculator_tool(), note_tool(NOTES_DIR)]

TOOLS = build_tools(ENGINE_B)
TOOLS_BY_NAME = {t.name: t for t in TOOLS}

show_table(pd.DataFrame([{"tool": t.name, "parameters": ", ".join(t.parameters) or "—",
                          "type": "✍️ write" if t.writes else "👁️ read",
                          "in the interface": "🕒 human approval" if t.writes else "✅ executed",
                          "description": t.description} for t in TOOLS]),
           "In the interface, every compliant write goes through the human approval queue (Step 6).")

In [ ]:
# The guard, demonstrated: only arithmetic operations get through
calc = TOOLS_BY_NAME["calculate"]
trials = ["19.2 - 8.6", "1,000 / 4", "(88.5 - 50) * 2", "__import__('os').system('echo hacked')",
          "open('/etc/passwd').read()", "9 ** 999999"]
show_table(pd.DataFrame([{"expression received from the model": e, "result": calc.run(expression=e)} for e in trials]),
           "The last three attempts are refused without ever being executed.")

## Step 5 · The protocol: how a model “requests”

**🎯 Goal:** understand that a model does **nothing** by itself; it produces text that *describes* an action.

On every turn, the model replies with **a single JSON object**:

```json
{"thought": "I look for the rate", "tool": "search_documents", "args": {"query": "youth unemployment"}}
```

or, to conclude:

```json
{"thought": "I have everything", "answer": "The rate is 19.2% [search_documents]."}
```

**🔍 What to notice:** the parser is **tolerant** of what models actually produce (JSON wrapped in text or code fences) but **strict** about everything else. An unreadable reply does not stop the agent: the error is sent back to the model so it can correct itself.

In [ ]:
# ── Step 5 · System instructions and parser ─────────────────────────────────
def describe_tools(tools):
    lines = []
    for t in tools:
        params = ", ".join(f"{p}: {d}" for p, d in t.parameters.items()) or "none"
        lines.append(f"- {t.name}({params}) — {t.description}{' [WRITE: subject to approval]' if t.writes else ''}")
    return "\n".join(lines)

AGENT_INSTRUCTIONS = """You are an analysis agent for a national statistical office.
Available tools:
<<TOOLS>>

Format: on EVERY turn, reply with ONE JSON object and nothing else.
- To call a tool: {"thought": "...", "tool": "<name>", "args": {...}}
- To conclude: {"thought": "...", "answer": "..."}
Rules:
1. Use search_documents before stating anything about the data.
2. Use calculate for any arithmetic.
3. Cite in square brackets the tool each figure comes from, e.g. [search_documents].
4. If a tool is refused, do not retry: explain it in your answer.
5. NEVER quote a figure that a tool did not return."""

print(AGENT_INSTRUCTIONS.replace("<<TOOLS>>", describe_tools(TOOLS)))

In [ ]:
class ProtocolError(Exception):
    """The model's reply cannot be interpreted as an action."""

@dataclass
class Action:
    thought: str = ""
    tool: Optional[str] = None
    args: dict = field(default_factory=dict)
    answer: Optional[str] = None

    @property
    def is_final(self):
        return self.answer is not None

def extract_json(text):
    """Find the first balanced JSON object, even when wrapped in text or ``` fences."""
    start = text.find("{")
    while start != -1:
        depth, in_string, escaped = 0, False, False
        for i in range(start, len(text)):
            c = text[i]
            if in_string:
                escaped = (c == "\\") and not escaped
                if c == '"' and not escaped:
                    in_string = False
                continue
            if c == '"':
                in_string = True
            elif c == "{":
                depth += 1
            elif c == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start:i + 1])
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    raise ProtocolError("no usable JSON object in the reply")

def parse_action(text):
    obj = extract_json(text)
    if "answer" in obj:
        return Action(thought=str(obj.get("thought", "")), answer=str(obj["answer"]))
    if "tool" in obj:
        args = obj.get("args", {})
        if not isinstance(args, dict):
            raise ProtocolError("“args” must be a JSON object")
        return Action(thought=str(obj.get("thought", "")), tool=str(obj["tool"]), args=args)
    raise ProtocolError("the JSON object contains neither “tool” nor “answer”")

# Tolerant of what models really produce, strict about the rest
EXAMPLES = [
    '{"thought":"searching","tool":"search_documents","args":{"query":"unemployment"}}',
    '```json\n{"tool":"calculate","args":{"expression":"2+2"}}\n```',
    'Sure! {"answer":"8.6% [search_documents]"} Let me know if…',
    '{"tool":"calculate","args":"19.2-8.6"}',
    'I will now search the documents.',
]
rows = []
for text in EXAMPLES:
    try:
        a = parse_action(text)
        rows.append({"raw model reply": text.replace("\n", " ⏎ "), "verdict": "✅ accepted",
                     "interpretation": f"answer: {a.answer}" if a.is_final else f"tool: {a.tool} {a.args}"})
    except ProtocolError as e:
        rows.append({"raw model reply": text, "verdict": "❌ rejected", "interpretation": str(e)})
show_table(pd.DataFrame(rows), "A rejection does not stop the agent: the error is sent back to the model so it can correct itself.")

## Step 6 · Policies and approval queue

**🎯 Goal:** put human control **in the right place**.

Between the model’s **request** and the tool’s **execution**, one function decides: the **policy**. It returns “yes” or “no”, **with a logged reason**.

```python
allowed, reason = policy(tool, arguments)   # ← “the gap”
if not allowed:
    log(reason); continue                   # the tool is never called
result = tool.run(**arguments)
```

**⭐ New in this edition: the approval queue.** In an interface, you cannot freeze the agent while waiting for a human. A compliant write is therefore **neither executed nor refused**: it is **queued**. An analyst then approves or rejects it from the ✅ *Approvals* tab of the interface. Every decision is time-stamped and chained in a log.

| Situation | Decision |
|:--|:--|
| Ordinary read | ✅ executed immediately |
| Request containing an individual identifier | ⛔ refused (statistical confidentiality) |
| Non-compliant write (file name, “fictional” label, length) | ⛔ refused |
| Compliant write | 🕒 **queued for approval** |

In [ ]:
# ── Step 6 · Approval policies ──────────────────────────────────────────────
def read_only_policy(tool, args):
    return (False, "write refused by default") if tool.writes else (True, "read allowed")

def allow_all_policy(tool, args):
    return True, "everything allowed (demonstration)"

def interactive_policy(tool, args):
    if not tool.writes:
        return True, "read allowed"
    reply = input(f"The agent wants to run {tool.name}({args}). Allow? [y/N] ")
    return (True, "approved by a human") if reply.strip().lower() == "y" else (False, "refused by a human")

# ⭐ A business policy that inspects the content of the request
IDENTIFIER_PATTERN = re.compile(r"\b(respondent[_\- ]?id|household number|national id|\d{9,})\b", re.IGNORECASE)
MAX_NOTE_LENGTH = 2000

def office_policy(tool, args):
    args_text = " ".join(str(v) for v in args.values())
    if IDENTIFIER_PATTERN.search(args_text):
        return False, "request about an individual identifier (statistical confidentiality)"
    if not tool.writes:
        return True, "read allowed"
    name = str(args.get("file_name", ""))
    if not re.fullmatch(r"[\w\-]{1,60}\.md", name):
        return False, f"non-compliant file name: {name!r}"
    if len(str(args.get("text", ""))) > MAX_NOTE_LENGTH:
        return False, "note too long"
    if "fictional" not in str(args.get("text", "")).lower():
        return False, "a note based on the fictional corpus must carry the label “fictional”"
    return True, "write compliant with the office policy"

# A policy is code: test it like code
cases = [
    ("search_documents", {"query": "youth unemployment rate"}),
    ("search_documents", {"query": "household income respondent_id 004512"}),
    ("save_note",        {"file_name": "finding.md", "text": "Gap of 10.6 points (fictional corpus)."}),
    ("save_note",        {"file_name": "../../etc/passwd", "text": "..."}),
    ("save_note",        {"file_name": "finding.md", "text": "Gap of 10.6 points."}),
]
rows = []
for tool_name, args in cases:
    row = {"tool": tool_name, "arguments": json.dumps(args, ensure_ascii=False)}
    for policy in (read_only_policy, allow_all_policy, office_policy):
        ok, reason = policy(TOOLS_BY_NAME[tool_name], args)
        row[policy.__name__.replace("_policy", "")] = f"{'✅' if ok else '⛔'} {reason}"
    rows.append(row)
show_table(pd.DataFrame(rows), "Same request, three policies: only the business policy blocks the data leak "
           "and the malicious path while allowing the compliant note.", max_width=70)

In [ ]:
# ── Step 6 (continued) · The human approval queue ───────────────────────────
@dataclass
class ApprovalRequest:
    number: int
    tool: str
    args: dict
    task: str
    created_at: str
    status: str = "pending"               # pending · approved · rejected
    decided_by: str = ""
    decided_at: str = ""
    result: str = ""


class ApprovalQueue:
    """Compliant writes wait for a human decision; every decision is hash-chained."""

    def __init__(self, tools_by_name):
        self.tools = tools_by_name
        self.requests, self.decisions = [], []
        self.current_task = ""

    def approval_queue_policy(self, tool, args):
        allowed, reason = office_policy(tool, args)
        if not allowed or not tool.writes:
            return allowed, reason
        request = ApprovalRequest(len(self.requests) + 1, tool.name, dict(args), self.current_task,
                                  datetime.now(timezone.utc).isoformat(timespec="seconds"))
        self.requests.append(request)
        return False, f"queued for approval (request #{request.number}): nothing is written before human validation"

    def pending(self):
        return [r for r in self.requests if r.status == "pending"]

    def decide(self, number, approve, decided_by="analyst"):
        request = next(r for r in self.requests if r.number == number)
        if request.status != "pending":
            return request
        if approve:
            request.result = self.tools[request.tool].run(**request.args)
            request.status = "approved"
        else:
            request.status = "rejected"
        request.decided_by = decided_by
        request.decided_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
        previous = self.decisions[-1]["hash"] if self.decisions else "0" * 64
        entry = {"number": number, "tool": request.tool, "args": request.args, "status": request.status,
                 "decided_by": decided_by, "timestamp": request.decided_at, "previous_hash": previous}
        entry["hash"] = hashlib.sha256(json.dumps(entry, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
        self.decisions.append(entry)
        return request


QUEUE = ApprovalQueue(TOOLS_BY_NAME)

# Demonstration: a compliant note is queued, then approved
note = TOOLS_BY_NAME["save_note"]
ok, reason = QUEUE.approval_queue_policy(note, {"file_name": "demo.md", "text": "Queue test (fictional corpus)."})
before = NOTES_DIR.exists() and any(NOTES_DIR.iterdir())
QUEUE.decide(1, approve=True)
after = sorted(p.name for p in NOTES_DIR.glob("*"))
show_table(pd.DataFrame([
    {"moment": "① the model requests the write", "decision": f"{'✅' if ok else '🕒'} {reason}", "file on disk": "yes" if before else "no"},
    {"moment": "② an analyst approves", "decision": QUEUE.requests[0].status, "file on disk": ", ".join(after)},
]), "The file only exists after the human decision. Check the disk, not just the log.")
(NOTES_DIR / "demo.md").unlink(missing_ok=True)
QUEUE = ApprovalQueue(TOOLS_BY_NAME)       # fresh queue for the interface

## Step 7 · The loop, grounding and audit

**🎯 Goal:** assemble the core of the agent and its two output controls.

**🔁 The loop** repeats three moves until a final answer or until the budget (`max_steps`) runs out:

1. **Parse** the model’s request (valid JSON? known tool?);
2. **Consult the policy** (allowed? refused? queued?);
3. **Execute and log**, then send the result back to the model.

**✅ The grounding check** verifies that every number in the final answer appears in the tool results, and that every cited source was actually called. It catches **invented figures**.

**🧾 The audit log** keeps what the model *actually wrote*. Each entry contains the SHA-256 fingerprint of the previous one: any later edit **breaks the chain**.

In [ ]:
# ── Step 7 · The agent ──────────────────────────────────────────────────────
@dataclass
class Step:
    number: int
    raw: str                                   # what the model actually wrote
    action: Optional[Action] = None
    allowed: Optional[bool] = None
    reason: str = ""
    result: Optional[str] = None
    error: Optional[str] = None
    duration_ms: float = 0.0

@dataclass
class Run:
    task: str
    policy: str
    steps: list = field(default_factory=list)
    answer: str = ""
    stop: str = ""

    def log(self):
        rows = []
        for s in self.steps:
            a = s.action
            rows.append({
                "step": s.number,
                "request": ("final answer" if a and a.is_final else (a.tool if a else "— unreadable —")),
                "arguments": json.dumps(a.args, ensure_ascii=False) if a and not a.is_final else "",
                "decision": "" if s.allowed is None else ("✅ allowed" if s.allowed else
                                                          ("🕒 pending" if s.reason.startswith("queued") else "⛔ refused")),
                "reason / error": s.error or s.reason,
                "result": (s.result or "").replace("\n", " ")[:90],
                "ms": round(s.duration_ms, 1),
            })
        return pd.DataFrame(rows)


class ScriptedModel:
    """Replays predefined replies. No decisions, no API key."""
    def __init__(self, replies):
        self.replies, self.calls = list(replies), 0
    def __call__(self, instructions, messages):
        self.calls += 1
        return self.replies.pop(0) if self.replies else '{"answer": "(end of script)"}'


class Agent:
    def __init__(self, tools, model, policy=read_only_policy, max_steps=8):
        self.tools = {t.name: t for t in tools}
        self.model, self.policy, self.max_steps = model, policy, max_steps
        self.instructions = AGENT_INSTRUCTIONS.replace("<<TOOLS>>", describe_tools(tools))

    def run(self, task, verbose=True):
        run = Run(task=task, policy=self.policy.__name__)
        messages = [{"role": "user", "content": task}]
        say = print if verbose else (lambda *a, **k: None)
        say(f"🎯 TASK: {task}\n")

        for n in range(1, self.max_steps + 1):
            t0 = time.perf_counter()
            raw = self.model(self.instructions, messages)
            messages.append({"role": "assistant", "content": raw})
            step = Step(number=n, raw=raw)
            run.steps.append(step)

            # ① Parse
            try:
                action = parse_action(raw)
                step.action = action
            except ProtocolError as e:
                step.error = f"protocol: {e}"
                messages.append({"role": "user", "content": f"PROTOCOL ERROR: {e}. Reply with a JSON object only."})
                say(f"  {n}. ❌ unreadable reply → error sent back to the model")
                continue

            if action.is_final:
                run.answer, run.stop = action.answer, "final answer"
                say(f"  {n}. 🏁 final answer")
                break

            tool = self.tools.get(action.tool)
            if tool is None:
                step.error = f"unknown tool: {action.tool}"
                messages.append({"role": "user", "content": f"ERROR: the tool “{action.tool}” does not exist. "
                                 f"Valid tools: {', '.join(self.tools)}."})
                say(f"  {n}. ❓ unknown tool “{action.tool}” → error sent back")
                continue

            # ② Policy: the gap
            step.allowed, step.reason = self.policy(tool, action.args)
            if not step.allowed:
                status = "PENDING HUMAN APPROVAL" if step.reason.startswith("queued") else "REFUSED"
                messages.append({"role": "user", "content": f"RESULT OF {tool.name}: {status} — {step.reason}."})
                say(f"  {n}. {'🕒' if status.startswith('PENDING') else '⛔'} {tool.name}: {step.reason}")
                continue

            # ③ Execute
            try:
                step.result = tool.run(**action.args)
            except TypeError as e:
                step.error = f"invalid arguments: {e}"
            step.duration_ms = (time.perf_counter() - t0) * 1000
            feedback = step.result if step.error is None else f"ERROR: {step.error}"
            messages.append({"role": "user", "content": f"RESULT OF {tool.name}:\n{feedback}"})
            say(f"  {n}. 🛠️ {tool.name}({json.dumps(action.args, ensure_ascii=False)[:60]}) → {feedback.splitlines()[0][:70]}")
        else:
            run.stop = f"budget exhausted ({self.max_steps} steps)"
            run.answer = "Stopped: budget reached without a final answer."
            say(f"  ⏹️ {run.stop}")

        say(f"\n💬 ANSWER: {run.answer}")
        return run


# ✅ Grounding check
_NUMBER = re.compile(r"\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?")

def _numbers(text):
    return {float(n.replace(",", "")) for n in _NUMBER.findall(text)}

def check_grounding(run):
    observations = " ".join(s.result or "" for s in run.steps)
    known = _numbers(observations)
    called = {s.action.tool for s in run.steps if s.action and not s.action.is_final and s.result}
    rows = [{"item": f"number {n:g}", "status": "✅ grounded" if n in known else "🚩 not found in tool results"}
            for n in sorted(_numbers(run.answer))]
    for block in re.findall(r"\[([^\]]+)\]", run.answer):
        for source in (s.strip() for s in block.split(",")):
            rows.append({"item": f"source [{source}]",
                         "status": "✅ tool actually called" if source in called else "🚩 source never called"})
    report = pd.DataFrame(rows)
    return report, (not report.empty and report["status"].str.startswith("✅").all())


# 🧾 Hash-chained audit log
def _fingerprint(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode("utf-8")).hexdigest()

def save_log(run, path, meta):
    previous, entries = "0" * 64, []
    for s in run.steps:
        a = s.action
        entry = {
            "step": s.number, "raw": s.raw,
            "tool": (a.tool if a and not a.is_final else None), "args": (a.args if a and not a.is_final else None),
            "final": bool(a and a.is_final), "allowed": s.allowed, "reason": s.reason,
            "result": s.result, "error": s.error, "previous_hash": previous,
        }
        entry["hash"] = previous = _fingerprint(entry)
        entries.append(entry)
    report, grounded = check_grounding(run)
    document = {
        "meta": {**meta, "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                 "task": run.task, "policy": run.policy, "stop": run.stop,
                 "answer": run.answer, "fully_grounded": bool(grounded)},
        "steps": entries,
        "seal": previous,
    }
    Path(path).write_text(json.dumps(document, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def verify_log(path):
    document = json.loads(Path(path).read_text(encoding="utf-8"))
    previous = "0" * 64
    for entry in document["steps"]:
        copy = {k: v for k, v in entry.items() if k != "hash"}
        if copy["previous_hash"] != previous or _fingerprint(copy) != entry["hash"]:
            return False, f"chain broken at step {entry['step']}"
        previous = entry["hash"]
    return (previous == document["seal"]), ("intact" if previous == document["seal"] else "invalid seal")

callout("Loop, grounding check and hash-chained log are ready.", "Agent core assembled", "success")

---
# 🟪 PART 4 — The “brains”

## Step 8 · Local demo or free model

**🎯 Goal:** understand that the brain is **interchangeable**. The agent, tools, policy and log do not change.

| Brain | Key | Quota | When to use it |
|:--|:--:|:--|:--|
| 🧪 **Local demo** (rules) | none | unlimited | Training, testing, offline demos |
| ⚡ **Groq** · `llama-3.1-8b-instant` | `GROQ_API_KEY` | ≈ 14,400 requests/day* | The most generous free quota, very fast |
| 🔷 **Gemini** · `gemini-flash-lite-latest` | `GEMINI_API_KEY` | a few hundred/day* | More capable model, long context |
| 🖥️ **Ollama** · `qwen2.5:3b` | none | unlimited | Sensitive data: nothing leaves the machine |

*\* Quotas published in 2026 and subject to change: check the “Limits” page of your account.*

**🔑 Adding a key in Colab:** click the **🔑 Secrets** icon in the left sidebar → **Add new secret** → name `GROQ_API_KEY` → paste the key → switch on **Notebook access**. Get a free key at `console.groq.com`.

**🧪 The demo brain** applies simple rules (search; calculate if a gap is requested; prepare a note if one is requested), but it replies **in the same JSON format** as a real model. The whole control chain is therefore genuinely exercised.

> ⚠️ **Free does not mean unconditional.** Before sending real statistical data to an online service, read its terms of use. For confidential data, prefer Ollama.

In [ ]:
# ── Step 8 · The demo brain (no network, same protocol) ─────────────────────
class RuleBasedModel:
    """Rule-based planner: it speaks the JSON protocol exactly like an LLM."""
    provider, model = "demo", "local rules"

    def __init__(self):
        self.calls, self.retries, self.tokens = 0, 0, 0

    def __repr__(self):
        return "🧪 demo · local rules"

    @staticmethod
    def _reply(thought, **action):
        return json.dumps({"thought": thought, **action}, ensure_ascii=False)

    def __call__(self, instructions, messages):
        self.calls += 1
        task = messages[0]["content"]
        t = _norm(task)
        feedback = [m["content"] for m in messages[1:] if m["role"] == "user"]
        def result(name):
            return next((f.split("\n", 1)[1] if "\n" in f else f for f in feedback if f.startswith(f"RESULT OF {name}")), None)

        wants_list = any(k in t for k in ("which documents", "what documents", "list", "available publications",
                                           "how many documents", "how many publications"))
        wants_gap = any(k in t for k in ("gap", "difference", "exceed", "compare", "higher than", "lower than", "how much more"))
        wants_note = any(k in t for k in ("note", "save", "record", "memo", "write up"))
        topic = [w for w in tokenize(task) if w not in ("document", "publication", "available", "list", "note", "save")]

        # ① Inventory requested
        if wants_list and result("list_documents") is None:
            return self._reply("The user asks for the list of publications.", tool="list_documents", args={})
        # ② Search, as soon as there is a topic
        if topic and result("search_documents") is None:
            query = " ".join(w for w in topic if w not in INSTRUCTION_WORDS) if wants_note else task
            return self._reply("I search for relevant passages.", tool="search_documents", args={"query": query or task})
        passage = result("search_documents") or ""
        if passage.startswith("RESULT OF") or passage.startswith("ERROR"):
            reason = passage.split("—", 1)[-1].strip() if "—" in passage else passage
            return self._reply("My search was blocked: I explain why.", answer=f"⛔ I cannot process this request: {reason}")
        found = bool(passage) and not passage.startswith("NO RELEVANT PASSAGE")
        # ③ Gap between the first two percentages of the best passage
        percentages = re.findall(r"(\d+(?:\.\d+)?)\s?%", passage.split("\n")[0]) if found else []
        if wants_gap and len(percentages) >= 2 and result("calculate") is None:
            a, b = sorted(float(p) for p in percentages[:2])
            return self._reply("I compute the gap with the tool instead of estimating it.",
                               tool="calculate", args={"expression": f"{b} - {a}"})
        # ④ Note requested: prepared, then submitted to the policy
        if wants_note and found and result("save_note") is None:
            name = "note_" + "_".join(w for w in topic if w not in INSTRUCTION_WORDS)[:40] + ".md"
            items = "\n".join(f"- {line}" for line in passage.split("\n")[:2])
            text = f"Question: {task}\nKey elements:\n{items}\n(Source: fictional corpus of the STG17 workshop)"
            return self._reply("I prepare the requested note.", tool="save_note", args={"file_name": name, "text": text})

        # ⑤ Final answer, built only from tool results
        parts = []
        inventory = result("list_documents")
        if inventory:
            parts.append(f"**{inventory.split(chr(10))[0].rstrip(':')}** [list_documents].")
        if passage and wants_note and found:
            excerpts = []
            for line in passage.split("\n")[:2]:
                doc, _, text = line.partition("] ")
                excerpts.append(f"- “{text}” (`{doc.lstrip('[')}`)")
            parts.append("Key elements for the note [search_documents]:\n" + "\n".join(excerpts))
        elif passage:
            if found:
                doc, _, text = passage.split("\n")[0].partition("] ")
                parts.append(f"According to the publication `{doc.lstrip('[')}` [search_documents]:\n“{text}”")
            else:
                parts.append("I found **no reliable information** on this topic in the corpus [search_documents].")
        calculation = result("calculate")
        if calculation and not calculation.startswith("ERROR"):
            parts.append(f"➡️ The gap is **{calculation} percentage points** [calculate].")
        writing = result("save_note")
        if writing:
            if "PENDING" in writing:
                parts.append("📝 The note has been prepared and placed in the **approval queue**: "
                             "it will only be written once an analyst validates it.")
            else:
                parts.append(f"⛔ The note was not saved: {writing.split('—', 1)[-1].strip()}")
        if not parts:
            parts.append("Could you clarify your question? I can search the publications, list documents, "
                         "calculate a gap or prepare a note.")
        return self._reply("I have what I need.", answer="\n".join(parts))


callout("The demo brain is ready: it does not call any online service.", "Local brain", "success")

In [ ]:
# ── Step 8 (continued) · Connector to free models ───────────────────────────
PROVIDER = "auto"      # "auto", "groq", "gemini" or "ollama"
MODEL = None           # None = provider's default model; e.g. "openai/gpt-oss-120b"

PROVIDERS = {
    "groq":   {"url": "https://api.groq.com/openai/v1", "key": "GROQ_API_KEY",
               "model": "llama-3.1-8b-instant", "calls_per_minute": 30},
    "gemini": {"url": "https://generativelanguage.googleapis.com/v1beta/openai", "key": "GEMINI_API_KEY",
               "model": "gemini-flash-lite-latest", "calls_per_minute": 10},
    "ollama": {"url": os.environ.get("OLLAMA_URL", "http://localhost:11434") + "/v1", "key": None,
               "model": "qwen2.5:3b", "calls_per_minute": 600},
}

def read_secret(name):
    """Look for a key in environment variables, then in Colab Secrets."""
    if not name:
        return None
    value = os.environ.get(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    return value

def ollama_available():
    import requests
    try:
        return requests.get(PROVIDERS["ollama"]["url"].removesuffix("/v1") + "/api/tags", timeout=2).ok
    except Exception:
        return False


class ProviderError(Exception):
    pass


class OpenAICompatibleModel:
    """Adapter (instructions, messages) -> text for any /chat/completions API."""

    def __init__(self, provider="auto", model=None, temperature=0.0, max_tokens=700, max_retries=4):
        self.provider = self._choose(provider)
        conf = PROVIDERS[self.provider]
        self.url = conf["url"].rstrip("/") + "/chat/completions"
        self.key = read_secret(conf["key"]) if conf["key"] else "ollama"
        self.model = model or conf["model"]
        self.interval = 60.0 / conf["calls_per_minute"] * 1.05        # 5% safety margin
        self.temperature, self.max_tokens, self.max_retries = temperature, max_tokens, max_retries
        self.calls, self.retries, self.tokens, self._last = 0, 0, 0, 0.0

    @staticmethod
    def _choose(provider):
        if provider != "auto":
            conf = PROVIDERS[provider]
            if conf["key"] and not read_secret(conf["key"]):
                raise ProviderError(f"key {conf['key']} missing")
            if provider == "ollama" and not ollama_available():
                raise ProviderError("Ollama server unreachable")
            return provider
        for name in ("groq", "gemini"):
            if read_secret(PROVIDERS[name]["key"]):
                return name
        if ollama_available():
            return "ollama"
        raise ProviderError("no provider: add GROQ_API_KEY or GEMINI_API_KEY, or start Ollama")

    def __call__(self, instructions, messages):
        import requests
        body = {"model": self.model, "temperature": self.temperature, "max_tokens": self.max_tokens,
                "messages": [{"role": "system", "content": instructions}] + messages}
        for attempt in range(self.max_retries + 1):
            wait = self.interval - (time.monotonic() - self._last)      # rate limiter
            if wait > 0:
                time.sleep(wait)
            self._last = time.monotonic()
            self.calls += 1
            r = requests.post(self.url, json=body, timeout=120,
                              headers={"Authorization": f"Bearer {self.key}", "Content-Type": "application/json"})
            if r.status_code in (429, 500, 502, 503) and attempt < self.max_retries:
                try:
                    delay = float(r.headers.get("retry-after") or 0)
                except ValueError:
                    delay = 0
                delay = min(90, delay or 2 ** (attempt + 1))
                self.retries += 1
                cause = "quota reached" if r.status_code == 429 else "server unavailable"
                print(f"     ⏳ HTTP {r.status_code} ({cause}) → retrying in {delay:.0f} s")
                time.sleep(delay)
                continue
            if not r.ok:
                raise ProviderError(f"HTTP {r.status_code}: {r.text[:200]}")
            data = r.json()
            self.tokens += (data.get("usage") or {}).get("total_tokens", 0) or 0
            return data["choices"][0]["message"].get("content") or ""
        raise ProviderError("quota still reached after several attempts")

    def __repr__(self):
        return f"{self.provider} · {self.model}"


show_table(pd.DataFrame([{"provider": n, "key variable": c["key"] or "— (local)",
                          "available": "✅" if (read_secret(c["key"]) if c["key"] else ollama_available()) else "—",
                          "default model": c["model"], "target calls/min": c["calls_per_minute"]}
                         for n, c in PROVIDERS.items()]),
           "Automatic detection of available providers.")

In [ ]:
# ── Optional · Install Ollama on Colab (local model, unlimited, no key) ────
INSTALL_OLLAMA = False   # set to True, then run the cell (≈ 2 to 4 minutes)

if INSTALL_OLLAMA:
    if not shutil.which("ollama"):
        subprocess.run("apt-get -qq install -y zstd > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=False)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    subprocess.run(["ollama", "pull", PROVIDERS["ollama"]["model"]], check=False)
    callout(f"Ollama available: <b>{ollama_available()}</b>. On Colab, a GPU runtime "
            "(Runtime → Change runtime type) makes replies much faster.", "Ollama", "info")
else:
    callout("Ollama is not installed. Set <code>INSTALL_OLLAMA = True</code> for a local, unlimited and "
            "confidential model — with no key at all.", "Local option", "info")

## Step 9 · Text-mode trial

**🎯 Goal:** check that everything works **before** opening the interface.

The `ask()` function produces exactly the same display as the interface. It is useful wherever interactive widgets do not render (GitHub preview, some editors).

```python
ask("What is the youth unemployment rate?")
ask("What was inflation in March 2024?", mode="rag", engine="A · Naive")
```

In [ ]:
# ── Step 9 · Answer formatting and text mode ────────────────────────────────
SOURCE_ICONS = {"search_documents": "🔎", "calculate": "🧮", "list_documents": "📚", "save_note": "📝"}

def _chip(source):
    icon = SOURCE_ICONS.get(source, "📄")
    return (f"<span style='display:inline-block;background:#E8F5EF;color:#0F5A31;border:1px solid #BFE3CE;"
            f"border-radius:12px;padding:0 8px;margin:0 2px;font-size:11.5px;white-space:nowrap'>{icon} {source}</span>")

def text_to_html(text):
    """Convert the model's text (light Markdown) into safe, readable HTML."""
    lines, bullets = [], []
    def flush():
        if bullets:
            lines.append("<ul style='margin:4px 0 4px 18px;padding:0'>" + "".join(f"<li>{b}</li>" for b in bullets) + "</ul>")
            bullets.clear()
    for raw in str(text).strip().splitlines():
        line = html.escape(raw.strip())
        line = re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", line)
        line = re.sub(r"(?<![\w*])\*(?!\s)(.+?)\*(?!\w)", r"<i>\1</i>", line)
        line = re.sub(r"`([^`]+)`", r"<code style='background:#EEF2F0;color:#0B2545;padding:1px 5px;border-radius:4px'>\1</code>", line)
        line = re.sub(r"\[([A-Za-z0-9_ ,\-]{3,80})\]", lambda m: "".join(_chip(s.strip()) for s in m.group(1).split(",")), line)
        if re.match(r"^(-|\*|•)\s+", raw.strip()):
            bullets.append(re.sub(r"^(-|\*|•)\s+", "", line))
            continue
        flush()
        if raw.strip().startswith("#"):
            line = f"<b style='color:#0B2545'>{line.lstrip('#').strip()}</b>"
        lines.append(line if line else "<div style='height:6px'></div>")
    flush()
    out = ""
    for i, l in enumerate(lines):
        is_block = l.startswith("<ul")
        after_block = i > 0 and lines[i - 1].startswith("<ul")
        out += ("" if i == 0 or is_block or after_block else "<br>") + l
    return out

def df_to_html(df):
    head = "".join(f"<th style='background:#0B2545;color:#fff;padding:5px 8px;text-align:left;font-weight:600'>{html.escape(str(c))}</th>" for c in df.columns)
    body = ""
    for i, (_, row) in enumerate(df.iterrows()):
        bg = "#F4F7F5" if i % 2 else "#FFFFFF"
        body += "<tr>" + "".join(f"<td style='padding:5px 8px;border-bottom:1px solid #DDE3E0;background:{bg};vertical-align:top'>"
                                 f"{html.escape(str(v))}</td>" for v in row.values) + "</tr>"
    return (f"<div style='overflow-x:auto'><table style='border-collapse:collapse;font-size:12px;width:100%;"
            f"font-family:Segoe UI,system-ui,sans-serif;color:#1d2b24'><tr>{head}</tr>{body}</table></div>")

FONT = "font-family:Segoe UI,system-ui,-apple-system,sans-serif"

def user_bubble(question):
    return (f"<div style='display:flex;justify-content:flex-end;margin:10px 0'>"
            f"<div style='max-width:78%;background:#0B2545;color:#fff;padding:10px 14px;border-radius:14px 14px 2px 14px;"
            f"{FONT};font-size:14px;line-height:1.5'>🧑‍💼 {html.escape(question)}</div></div>")

def agent_bubble(text, meta="", details_html="", colour=GREEN):
    details = (f"<details style='margin-top:8px'><summary style='cursor:pointer;color:#12507A;font-size:12.5px'>"
               f"🧭 Show reasoning and checks</summary><div style='margin-top:6px'>{details_html}</div></details>"
               if details_html else "")
    return (f"<div style='display:flex;justify-content:flex-start;margin:10px 0'>"
            f"<div style='max-width:86%;background:#FFFFFF;color:#1d2b24;border:1px solid #DDE3E0;border-left:5px solid {colour};"
            f"padding:10px 14px;border-radius:14px 14px 14px 2px;{FONT};font-size:14px;line-height:1.6;box-shadow:0 1px 3px rgba(0,0,0,.06)'>"
            f"<div>🤖 {text_to_html(text)}</div>"
            f"<div style='margin-top:8px;color:#6B7B75;font-size:11.5px'>{meta}</div>{details}</div></div>")

def system_bubble(text, colour=GOLD):
    return (f"<div style='margin:8px auto;max-width:90%;text-align:center;background:#FFF7E0;border:1px dashed {colour};"
            f"color:#5c4500;padding:6px 12px;border-radius:10px;{FONT};font-size:12.5px'>{text}</div>")


def process_question(question, brain, mode="agent", engine="B · Improved", max_steps=6, queue=None):
    """Single entry point, shared by text mode and the interface."""
    queue = queue or QUEUE
    engine_obj = ENGINES[engine]
    start = time.perf_counter()
    calls_before = brain.calls
    if mode == "rag":
        generator = None if isinstance(brain, RuleBasedModel) else brain
        r = answer(question, engine=engine_obj, generator=generator)
        return {"question": question, "answer": r["answer"], "mode": "Simple RAG", "run": None,
                "grounding": None, "grounded": None, "steps": 0, "duration": time.perf_counter() - start,
                "calls": brain.calls - calls_before,
                "details": f"<div style='{FONT};font-size:12.5px'>Mode: <b>{r['mode']}</b> · best-passage confidence: "
                           f"<b>{r['confidence']:.3f}</b> · rule: cosine ≥ {COSINE_THRESHOLD} or coverage ≥ {COVERAGE_THRESHOLD}"
                           f" · engine: <b>{engine}</b></div>"}
    tools = build_tools(engine_obj)
    queue.current_task = question
    agent = Agent(tools, brain, policy=queue.approval_queue_policy, max_steps=max_steps)
    run = agent.run(question, verbose=False)
    report, grounded = check_grounding(run)
    details = (f"<div style='{FONT};font-size:12.5px;margin-bottom:4px'><b>Agent trace</b> · policy: "
               f"<code>{run.policy}</code> · stop: {run.stop}</div>" + df_to_html(run.log().drop(columns=["ms"]))
               + f"<div style='{FONT};font-size:12.5px;margin:8px 0 4px'><b>Grounding check</b></div>"
               + (df_to_html(report) if not report.empty else "<i>no number or source to check</i>"))
    return {"question": question, "answer": run.answer, "mode": "Agent", "run": run, "grounding": report,
            "grounded": grounded, "steps": len(run.steps), "duration": time.perf_counter() - start,
            "calls": brain.calls - calls_before, "details": details}


def answer_meta(res, brain):
    badge = "" if res["grounded"] is None else (" · ✅ figures grounded" if res["grounded"] else " · 🚩 needs review")
    return (f"{res['mode']} · {brain!r} · {res['steps']} step(s) · {res['calls']} model call(s) · "
            f"{res['duration']:.1f} s{badge}")


DEMO_BRAIN = RuleBasedModel()

def ask(question, mode="agent", engine="B · Improved", brain=None, max_steps=6):
    """Text mode: same processing and same rendering as the interface."""
    brain = brain or DEMO_BRAIN
    res = process_question(question, brain, mode, engine, max_steps)
    display(HTML(user_bubble(question) + agent_bubble(res["answer"], answer_meta(res, brain), res["details"],
                                                      colour=GREEN if res["grounded"] in (True, None) else RED)))
    return res

_ = ask("By how much does youth unemployment exceed the overall rate?")
_ = ask("What is the US dollar exchange rate?")

In [ ]:
# ── Step 9 (continued) · Self-test: the full chain on six typical requests ──
tests = [
    ("What was inflation in March 2024?", "4.1"),
    ("What is the youth unemployment rate?", "19.2"),
    ("By how much does youth unemployment exceed the overall rate?", "10.6"),
    ("How many documents are available?", "9"),
    ("What is the US dollar exchange rate?", "no reliable information"),
    ("Prepare a note on cocoa production", "approval queue"),
]
rows = []
test_queue = ApprovalQueue(TOOLS_BY_NAME)
for question, expected in tests:
    res = process_question(question, RuleBasedModel(), queue=test_queue)
    passed = expected.lower() in res["answer"].lower()
    rows.append({"request": question, "expected": expected, "result": "✅" if passed else "❌",
                 "steps": res["steps"], "grounding": "✅" if res["grounded"] else ("—" if res["grounded"] is None else "🚩")})
test_summary = pd.DataFrame(rows)
show_table(test_summary, f"{(test_summary['result'] == '✅').sum()}/{len(tests)} requests handled as expected. "
           f"Write requests queued during the test: {len(test_queue.pending())} (no file written).")

---
# 🟥 PART 5 — Use the assistant

## Step 10 · The chat interface

**🎯 Goal:** put the agent in users’ hands **without losing any of the controls**.

### 🧑‍🏫 User guide

| Area | Role |
|:--|:--|
| **⚙️ Settings** | Choose the brain, the mode (agent or simple RAG), the search engine (A or B) and the step budget |
| **💡 Suggestions** | One click fills in the question |
| **💬 Conversation** | Your questions on the right, answers on the left, with their sources as chips 🔎 🧮 📚 📝 |
| **🧭 Trace** | Every step of the latest answer: request, decision, result |
| **✅ Approvals** | Pending writes: **Approve** or **Reject** |
| **🧾 Log** | Saved audit logs, their integrity, and conversation export |
| **📊 Statistics** | Questions, model calls, refusals, grounding rate |

### 🔬 Experiments to try

1. Ask **the same question** with engine **A** and then **B**: compare the source retrieved.
2. Ask **“Prepare a note on inflation”**, then open the **✅ Approvals** tab.
3. Ask an **off-corpus** question (“What is the price of oil?”): the assistant must refuse to invent.
4. Set the **budget** to 2 steps and ask for a gap: what happens?
5. With a Groq key, compare the **local demo** with the **real model**: which one follows the rules better?

> ℹ️ **If the interface does not show** (GitHub preview, editor without widgets), use `ask("your question")` from Step 9.

In [ ]:
# ── Step 10 · The interface ─────────────────────────────────────────────────
import warnings
import ipywidgets as widgets

BRAINS = [("🧪 Local demo (no key)", "demo"), ("⚡ Groq — free", "groq"),
          ("🔷 Gemini — free", "gemini"), ("🖥️ Ollama — local", "ollama")]
SUGGESTIONS = [
    "What is the youth unemployment rate?",
    "By how much does youth unemployment exceed the overall rate?",
    "What was inflation in March 2024?",
    "How many documents are available?",
    "Prepare a note on cocoa production",
    "What is the price of oil?",
]


class AssistantInterface:
    def __init__(self, queue):
        self.queue = queue
        self.thread = []                    # HTML blocks of the conversation
        self.results = []
        self.logs = []
        self.brain_cache = {"demo": RuleBasedModel()}
        self.folder = OUTPUTS / "interface"
        self.folder.mkdir(parents=True, exist_ok=True)
        self._build()
        self._welcome()

    # ── Layout ─────────────────────────────────────────────────────────────
    def _build(self):
        L = widgets.Layout
        header = widgets.HTML(
            f"<div style='background:linear-gradient(135deg,#0B2545 0%,#12507A 50%,#1B7A43 100%);border-radius:14px;"
            f"padding:14px 20px;{FONT}'><div style='color:#F2A900;font-size:11px;letter-spacing:2.5px;font-weight:700'>"
            f"NATIONAL STATISTICAL OFFICE · {COUNTRY['name'].upper()} · FICTIONAL DATA</div>"
            f"<div style='color:#fff;font-size:21px;font-weight:800;margin-top:3px'>🤖 Statistical Assistant</div>"
            f"<div style='color:#d6e4ef;font-size:13px'>Ask a question about the publications. Every figure is sourced, "
            f"every action is controlled, every exchange is logged.</div></div>")

        style = {"description_width": "90px"}
        self.w_brain = widgets.Dropdown(options=BRAINS, value="demo", description="🧠 Brain", style=style, layout=L(width="300px"))
        self.w_mode = widgets.Dropdown(options=[("🤖 Agent (tools)", "agent"), ("📄 Simple RAG", "rag")], value="agent",
                                       description="🎛️ Mode", style=style, layout=L(width="260px"))
        self.w_engine = widgets.Dropdown(options=list(ENGINES), value="B · Improved", description="🔎 Engine",
                                         style=style, layout=L(width="250px"))
        self.w_budget = widgets.IntSlider(value=6, min=2, max=10, description="⏱️ Budget", style=style, layout=L(width="300px"))
        settings = widgets.VBox([widgets.HBox([self.w_brain, self.w_mode]), widgets.HBox([self.w_engine, self.w_budget])])

        buttons = []
        for s in SUGGESTIONS:
            b = widgets.Button(description=s if len(s) <= 46 else s[:44] + "…", tooltip=s, layout=L(width="auto", margin="2px"))
            b.on_click(lambda _, q=s: self._suggest(q))
            buttons.append(b)
        suggestions = widgets.HBox(buttons, layout=L(flex_flow="row wrap"))

        self.w_chat = widgets.HTML(layout=L(width="100%"))
        self.w_question = widgets.Text(placeholder="Type your question and press Enter…", layout=L(width="70%"))
        self.w_send = widgets.Button(description="Send", icon="paper-plane", button_style="success", layout=L(width="130px"))
        self.w_clear = widgets.Button(description="Clear", icon="trash", layout=L(width="110px"))
        self.w_send.on_click(self._send)
        self.w_clear.on_click(self._clear)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            try:
                self.w_question.on_submit(self._send)
            except Exception:
                pass
        input_row = widgets.HBox([self.w_question, self.w_send, self.w_clear])
        self.w_status = widgets.HTML()

        self.w_trace = widgets.HTML()
        self.w_approvals = widgets.VBox()
        self.w_log = widgets.HTML()
        self.w_export = widgets.Button(description="Export conversation", icon="download", layout=L(width="240px"))
        self.w_export.on_click(self._export)
        self.w_export_msg = widgets.HTML()
        self.w_stats = widgets.HTML()
        self.tabs = widgets.Tab(children=[self.w_trace, self.w_approvals,
                                          widgets.VBox([self.w_log, widgets.HBox([self.w_export, self.w_export_msg])]),
                                          self.w_stats])
        for i, title in enumerate(["🧭 Trace", "✅ Approvals", "🧾 Log", "📊 Statistics"]):
            self.tabs.set_title(i, title)

        def area_title(t):
            return widgets.HTML(f"<div style='{FONT};font-weight:700;color:#0B2545;margin:10px 0 2px'>{t}</div>")

        self.app = widgets.VBox([header, area_title("⚙️ Settings"), settings, area_title("💡 Suggestions"), suggestions,
                                 area_title("💬 Conversation"), self.w_chat, input_row, self.w_status,
                                 area_title("🔬 Controls and traceability"), self.tabs],
                                layout=L(width="100%", max_width="1000px"))

    # ── Rendering ──────────────────────────────────────────────────────────
    def _welcome(self):
        self.thread = [agent_bubble("Hello! I answer from the **office's publications** (fictional data).\n"
                                    "- I **cite** the source of every figure;\n"
                                    "- I **calculate** with a tool instead of estimating;\n"
                                    "- every **write** waits for an analyst's validation.\n"
                                    "Pick a suggestion or type your question.", "Assistant ready")]
        self._refresh()

    def _refresh(self):
        self.w_chat.value = (f"<div style='background:#F7FAF8;border:1px solid #DDE3E0;border-radius:12px;padding:6px 12px;"
                             f"height:430px;overflow-y:auto;{FONT}'>" + "".join(self.thread) + "</div>")
        self._refresh_approvals()
        self._refresh_log()
        self._refresh_stats()

    def _set_status(self, text, colour="#6B7B75"):
        self.w_status.value = f"<div style='{FONT};font-size:12.5px;color:{colour};margin:4px 2px'>{text}</div>"

    def _refresh_approvals(self):
        pending = self.queue.pending()
        blocks = [widgets.HTML(f"<div style='{FONT};font-size:13px;margin:6px 0'>"
                               f"<b>{len(pending)}</b> pending request(s) · {len(self.queue.decisions)} decision(s) taken</div>")]
        for r in pending:
            preview = html.escape(str(r.args.get("text", ""))[:220]).replace("\n", "<br>")
            card = widgets.HTML(
                f"<div style='{FONT};border:1px solid #F2A900;background:#FFFBEF;border-radius:10px;padding:8px 12px;font-size:12.5px'>"
                f"<b>Request #{r.number}</b> · <code>{r.tool}</code> → <code>{html.escape(str(r.args.get('file_name', '')))}</code>"
                f"<br><span style='color:#6B7B75'>Original question: {html.escape(r.task)}</span>"
                f"<div style='margin-top:6px;background:#fff;border:1px solid #EEE;border-radius:6px;padding:6px'>{preview}</div></div>",
                layout=widgets.Layout(width="68%"))
            ok = widgets.Button(description="Approve", icon="check", button_style="success", layout=widgets.Layout(width="120px"))
            ko = widgets.Button(description="Reject", icon="times", button_style="danger", layout=widgets.Layout(width="110px"))
            ok.on_click(lambda _, n=r.number: self._decide(n, True))
            ko.on_click(lambda _, n=r.number: self._decide(n, False))
            blocks.append(widgets.HBox([card, widgets.VBox([ok, ko])], layout=widgets.Layout(margin="4px 0")))
        if self.queue.decisions:
            history = pd.DataFrame([{"#": e["number"], "tool": e["tool"], "decision": e["status"],
                                     "by": e["decided_by"], "at (UTC)": e["timestamp"], "fingerprint": e["hash"][:12] + "…"}
                                    for e in self.queue.decisions])
            blocks.append(widgets.HTML(f"<div style='{FONT};font-size:12.5px;margin:8px 0 4px'><b>Hash-chained decision history</b></div>"
                                       + df_to_html(history)))
        self.w_approvals.children = blocks

    def _refresh_log(self):
        if not self.logs:
            self.w_log.value = f"<div style='{FONT};font-size:13px;color:#6B7B75'>No log yet.</div>"
            return
        rows = []
        for path, question in self.logs[-12:]:
            ok, state = verify_log(path)
            rows.append({"file": path.name, "question": question[:60], "integrity": f"{'🔒' if ok else '🚩'} {state}"})
        self.w_log.value = (f"<div style='{FONT};font-size:12.5px;margin:6px 0'>Folder: <code>{self.folder}</code></div>"
                            + df_to_html(pd.DataFrame(rows)))

    def _refresh_stats(self):
        agent_results = [r for r in self.results if r["mode"] == "Agent"]
        refused = sum(1 for r in agent_results for s in r["run"].steps if s.allowed is False)
        grounded = sum(1 for r in agent_results if r["grounded"])
        cards = [("Questions", len(self.results)), ("Model calls", sum(r["calls"] for r in self.results)),
                 ("Actions refused or pending", refused),
                 ("Grounded answers", f"{grounded}/{len(agent_results)}" if agent_results else "—"),
                 ("Approved writes", sum(1 for r in self.queue.requests if r.status == "approved"))]
        self.w_stats.value = ("<div style='display:flex;flex-wrap:wrap;gap:10px;margin:8px 0'>" + "".join(
            f"<div style='{FONT};background:#fff;border:1px solid #DDE3E0;border-top:4px solid {GREEN};border-radius:10px;"
            f"padding:10px 16px;min-width:150px'><div style='font-size:24px;font-weight:800;color:#0B2545'>{v}</div>"
            f"<div style='font-size:12px;color:#6B7B75'>{t}</div></div>" for t, v in cards) + "</div>")

    # ── Actions ────────────────────────────────────────────────────────────
    def _brain(self):
        key = self.w_brain.value
        if key not in self.brain_cache:
            self.brain_cache[key] = OpenAICompatibleModel(key)
        return self.brain_cache[key]

    def _suggest(self, question):
        self.w_question.value = question

    def _send(self, _=None):
        question = self.w_question.value.strip()
        if not question:
            self._set_status("✍️ Type a question first.", GOLD)
            return
        self.w_question.value = ""
        self.w_send.disabled = True
        self.thread.append(user_bubble(question))
        self._refresh()
        self._set_status("⏳ The assistant is thinking…", "#12507A")
        try:
            brain = self._brain()
            res = process_question(question, brain, self.w_mode.value, self.w_engine.value, self.w_budget.value, self.queue)
            self.results.append(res)
            colour = GREEN if res["grounded"] in (True, None) else RED
            self.thread.append(agent_bubble(res["answer"], answer_meta(res, brain), res["details"], colour))
            self.w_trace.value = res["details"]
            if res["run"] is not None:
                path = self.folder / f"log_{len(self.logs) + 1:03d}.json"
                save_log(res["run"], path, {"country": COUNTRY["iso3"], "brain": repr(brain),
                                            "engine": self.w_engine.value, "budget": self.w_budget.value})
                self.logs.append((path, question))
                if any(s.reason.startswith("queued") for s in res["run"].steps):
                    self.thread.append(system_bubble("🕒 A write is waiting for your decision in the <b>✅ Approvals</b> tab."))
                    self.tabs.selected_index = 1
            self._set_status(f"✅ Answered in {res['duration']:.1f} s · {brain!r}", GREEN)
        except Exception as exc:
            self.thread.append(agent_bubble(f"I could not process the request: **{type(exc).__name__}** — {exc}\n"
                                            "Check the key of the selected brain, or switch back to the **local demo**.",
                                            "Error", colour=RED))
            self._set_status(f"⚠️ {type(exc).__name__}: {exc}", RED)
        finally:
            self.w_send.disabled = False
            self._refresh()

    def _decide(self, number, approve):
        r = self.queue.decide(number, approve)
        if approve:
            self.thread.append(system_bubble(f"✅ Request #{number} <b>approved</b> by the analyst — {html.escape(r.result)}", GREEN))
        else:
            self.thread.append(system_bubble(f"⛔ Request #{number} <b>rejected</b> by the analyst — no file written.", RED))
        self._refresh()

    def _clear(self, _=None):
        self.results.clear()
        self._welcome()
        self._set_status("🧹 Conversation cleared (audit logs are kept).")

    def _export(self, _=None):
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = self.folder / f"conversation_{stamp}.md"
        lines = [f"# Conversation — statistical assistant ({COUNTRY['name']}, fictional data)", ""]
        for r in self.results:
            lines += [f"## 🧑‍💼 {r['question']}", "", r["answer"], "",
                      f"*{r['mode']} · {r['steps']} step(s) · grounding: "
                      f"{'—' if r['grounded'] is None else ('complete' if r['grounded'] else 'needs review')}*", ""]
        path.write_text("\n".join(lines), encoding="utf-8")
        self.w_export_msg.value = f"<span style='{FONT};font-size:12.5px'>💾 <code>{path}</code></span>"
        try:
            from google.colab import files
            files.download(str(path))
        except Exception:
            pass

    def show(self):
        display(self.app)


APP = AssistantInterface(QUEUE)
APP.show()

> 💡 **Tip:** the interface stays active as long as the session runs. You can also drive it from code, e.g. `APP.w_question.value = "..."` then `APP._send()`.

## Step 11 · Under the hood

**🎯 Goal:** be able to explain to your management what happens when a user clicks **Send**.

**🔍 What to notice in the diagram:** the interface **never** talks to the tools directly. Everything goes through the agent loop and the policy; every write goes through a human; everything is logged.

In [ ]:
# ── Step 11 · Assistant architecture ────────────────────────────────────────
def _box(x, y, w, h, title, sub, border, fill="#FFFFFF"):
    return (f"<rect x='{x}' y='{y}' width='{w}' height='{h}' rx='12' fill='{fill}' stroke='{border}' stroke-width='2'/>"
            f"<text x='{x + w / 2}' y='{y + h / 2 - 4}' text-anchor='middle' font-size='13.5' font-weight='700' fill='#1d2b24'>{title}</text>"
            f"<text x='{x + w / 2}' y='{y + h / 2 + 14}' text-anchor='middle' font-size='11' fill='#56655e'>{sub}</text>")

def _arrow(x1, y1, x2, y2, colour="#56655e", label="", dy=-6):
    return (f"<line x1='{x1}' y1='{y1}' x2='{x2}' y2='{y2}' stroke='{colour}' stroke-width='2' marker-end='url(#ar)'/>"
            + (f"<text x='{(x1 + x2) / 2}' y='{(y1 + y2) / 2 + dy}' text-anchor='middle' font-size='10.5' fill='{colour}' font-weight='600'>{label}</text>" if label else ""))

svg = f"""<svg viewBox='0 0 940 400' width='100%' style='max-width:940px;background:#fff;{FONT}'>
<defs><marker id='ar' markerUnits='userSpaceOnUse' markerWidth='10' markerHeight='10' refX='9' refY='5' orient='auto'>
<path d='M0,0 L10,5 L0,10 z' fill='#56655e'/></marker></defs>
{_box(20, 40, 150, 64, "🧑‍💼 User", "asks a question", "#0B2545", "#E8EEF6")}
{_box(20, 160, 150, 64, "🖥️ Interface", "ipywidgets", "#0B2545", "#E8EEF6")}
{_box(240, 160, 170, 64, "🔁 Agent loop", "parse · decide · execute", GREEN, LIGHT)}
{_box(240, 30, 170, 64, "🧠 Brain", "demo · Groq · Gemini · Ollama", "#12507A", "#E8EEF6")}
{_box(480, 160, 150, 64, "🛡️ Policy", "logged reason", GOLD, "#FFF7E0")}
{_box(700, 90, 210, 64, "🔎 🧮 📚 Read tools", "executed immediately", GREEN, LIGHT)}
{_box(700, 230, 210, 64, "🕒 Approval queue", "pending writes", GOLD, "#FFF7E0")}
{_box(480, 310, 150, 64, "✅ Grounding", "figures checked", GREEN)}
{_box(240, 310, 170, 64, "🧾 Chained log", "SHA-256 · JSON", NAVY)}
{_box(20, 310, 150, 64, "👩‍💼 Analyst", "approves / rejects", "#0B2545", "#E8EEF6")}
{_arrow(95, 104, 95, 158, "#0B2545", "question", 0)}
{_arrow(170, 192, 238, 192, "#0B2545")}
{_arrow(325, 158, 325, 96, "#12507A", "instructions + history", 0)}
{_arrow(410, 192, 478, 192, GREEN, "request")}
{_arrow(630, 180, 698, 128, GREEN, "allowed", -2)}
{_arrow(630, 204, 698, 256, GOLD, "write", 14)}
{_arrow(325, 226, 325, 308, NAVY, "every step", 0)}
{_arrow(410, 226, 478, 318, GREEN, "answer")}
{_arrow(700, 280, 172, 338, GOLD, "to decide", -8)}
</svg>"""
display(HTML(svg))

show_table(pd.DataFrame([
    {"#": 1, "where": "Interface", "what happens": "the question is read and the settings are applied (brain, mode, engine, budget)"},
    {"#": 2, "where": "Loop", "what happens": "the brain receives the instructions and proposes an action in JSON"},
    {"#": 3, "where": "Parser", "what happens": "unreadable JSON or unknown tool → the error is sent back to the brain"},
    {"#": 4, "where": "Policy", "what happens": "read → executed; individual data → refused; compliant write → approval queue"},
    {"#": 5, "where": "Tools", "what happens": "the result is returned to the brain, which decides what to do next"},
    {"#": 6, "where": "Budget", "what happens": "beyond max_steps, the loop stops no matter what"},
    {"#": 7, "where": "Grounding", "what happens": "every number in the answer is looked up in the tool results"},
    {"#": 8, "where": "Log", "what happens": "raw replies and decisions are recorded and hash-chained"},
    {"#": 9, "where": "Analyst", "what happens": "approves or rejects writes; the decision is chained in turn"},
]), "The full journey of a question.")

---
# ⬛ PART 6 — Wrap up

## Step 12 · Limits, exercises and troubleshooting

### ⚠️ Limits of what you have built

- **The demo brain is not intelligent.** It applies fixed rules; it exercises the control chain, it does not judge answer quality.
- **Each question is handled independently.** The interface shows the history, but the agent does not use it: “and for women?” will not be understood.
- **The grounding check verifies numbers, not reasoning.** A correct figure can be attached to the wrong indicator.
- **The protocol is text-based.** Providers’ native tool calling would be more robust in production.
- **The interface runs inside a notebook.** For your colleagues, you will need a web application with authentication (Streamlit, Gradio, Dash…).

### ✅ Checkpoint

| Question | Answer |
|:--|:--|
| Why compare two search engines? | Because answer quality depends first on the passage retrieved |
| Why an approval **queue** in an interface? | You cannot freeze the agent while waiting for a human: the write waits, the agent answers |
| What proves a write did not happen? | **The disk**, not just the log |
| What does the grounding check detect? | Figures missing from tool results and sources that were never called |
| Why is the brain interchangeable? | Because all the control lives in **your** code, not in the model |

### 🧑‍💻 Your turn

1. **Conversation memory.** Change `process_question` to pass the last two exchanges to the brain. Test “and for women?”.
2. **New tool.** Add `compare_periods(indicator)` and justify, in two sentences, what a malicious request could do with it.
3. **Roles.** Add an “analyst” field to the interface and forbid the same person from both asking the question **and** approving the write.
4. **Quality.** Add a 👍 / 👎 button under each answer and record the feedback in the log.
5. **Production.** Port `AssistantInterface` to a Gradio or Streamlit app.

### 🛟 Troubleshooting

| Symptom | Fix |
|:--|:--|
| The interface does not show | Re-run the Step 10 cell; otherwise use `ask("…")` |
| “no provider” or “key missing” | Add `GROQ_API_KEY` in 🔑 Secrets and enable notebook access, or stay on the local demo |
| Repeated 429 errors | Free quota reached: wait, or switch brain |
| 404 “model not found” | Model names change: update `PROVIDERS[...]["model"]` in Step 8 |
| The real model invents a figure | A useful finding: the bubble turns red and the 🧭 Trace tab shows 🚩 grounding |
| Start from scratch | Menu **Runtime → Restart and run all** |

---

*STG17 Workshop · AfDB / STATAFRIC · “Agentic Statistical Assistant” · All data are fictional.*